# 08 Gold — Produtos Analíticos e Métricas Operacionais (Camada Enriquecida)

Este notebook implementa a camada **Gold** do pipeline de dados eleitorais (`tse-analysis-08-gold-analytics`), consolidando **5 tabelas analíticas** prontas para consumo sub-100ms em **DuckDB** e **Polars** (incluindo o consumidor **TS-Zé**):

1. **`gold_event_distribution`** — grão `(event_code, turn, uf)`: frequência absoluta, participação percentual e contagem de *clock drift* (`clock_drift_count`) de cada categoria de evento operacional da urna.
2. **`gold_session_completeness`** — grão `(turn, uf, final_state)`: taxa de encerramento regular (`CLOSED`) vs sessões incompletas (`INCOMPLETE`), contingências (`contingency_session_count`), ciclos de eleitores e total de eventos (com soma segura em `Int64`).
3. **`gold_biometric_attempts`** — grão `(turn, uf, biometric_attempt_num)`: distribuição ordinal e curva cumulativa de tentativas biométricas.
4. **`gold_section_turn_summary`** — grão `section_turn = (election_year, turn, uf, municipio_code, zone_num, section_num)` (~998 mil linhas): fato analítico central integrando métricas operacionais e biométricas da Silver com comparecimento oficial do Boletim de Urna (`CD_CARGO_PERGUNTA = 1`), perfil do eleitorado, local de votação georreferenciado, modelo de hardware da urna e divergências de correspondência.
5. **`gold_municipality_turn_summary`** — grão `(election_year, turn, uf, municipio_code)` (~11 mil linhas): visão consolidada municipal × turno para consultas instantâneas, rankings e mapas.

---
### Arquitetura de Execução em Escala
- **Passagem única em streaming sobre `fact_urna_event` (`4,95 bilhões` de linhas / 322 shards):** cada shard é copiado sequencialmente do Google Drive para o staging NVMe local (`/tmp/tse-eleicoes-staging`) com *prefetch* assíncrono (`ThreadPoolExecutor(max_workers=1)`), reduzido em memória com projeção mínima de colunas e descartado em seguida (pico de RAM `< 4 GB`).
- **Segurança Google Drive FUSE:** todas as gravações Gold ocorrem primeiro em staging local `/tmp` com compressão **ZSTD nível 5**, seguidas de limpeza do destino, cópia verificada por tamanho e validação independente via **DuckDB**.

In [1]:
import importlib
import importlib.util
import inspect
import os
import shutil
import stat
import subprocess
import sys
from pathlib import Path

_REQUIRED_PACKAGES = (
    ("kagglehub", "kagglehub>=0.4.0"),
    ("polars", "polars"),
    ("pyarrow", "pyarrow"),
    ("duckdb", "duckdb"),
)
if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    for _mod_name, _pip_name in _REQUIRED_PACKAGES:
        if importlib.util.find_spec(_mod_name) is None:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "-U", _pip_name]
            )
            importlib.invalidate_caches()
    import kagglehub as _kh_check

    if "output_dir" not in inspect.signature(_kh_check.dataset_download).parameters:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-U", "kagglehub>=0.4.0"]
        )
        importlib.invalidate_caches()
        sys.modules.pop("kagglehub", None)
        import kagglehub as _kh_check

# Contrato único de estado e ciclo de vida via KaggleHub SDK (Kaggle e Google Colab)
DATASET_HANDLE = os.getenv("KAGGLE_DATASET_HANDLE", "daniellmattos/tse-eleicoes-dados-v2")
DATASET_PATH = os.getenv("DATASET_PATH", "/content/drive/MyDrive/tse-eleicoes-dados/data")
STAGING_PATH = os.getenv("STAGING_PATH", "/tmp/tse-eleicoes-staging")
ALLOW_COLD_START = False


def is_directory_empty(path: Path) -> bool:
    """Verifica se DATASET_PATH está efetivamente vazio (ignorando arquivos ocultos de sistema)."""
    if not path.exists():
        return True
    return not any(item for item in path.iterdir() if not item.name.startswith("."))


def ensure_writable_tree(root: Path) -> None:
    """Garante permissão de escrita em árvores copiadas de mounts somente-leitura."""
    if not root.exists():
        return
    root.chmod(stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
    for dirpath, dirnames, filenames in os.walk(root):
        base = Path(dirpath)
        for dname in dirnames:
            dp = base / dname
            if not dp.is_symlink():
                dp.chmod(stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        for fname in filenames:
            fp = base / fname
            if not fp.is_symlink():
                fp.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IRGRP | stat.S_IROTH)


def is_dataset_not_found_error(exc: BaseException) -> bool:
    """Detecta erros de dataset inexistente (404 / Not Found) para permitir reconstrução do zero."""
    msg = f"{type(exc).__name__}: {exc}".lower()
    not_found_markers = (
        "404",
        "not found",
        "does not exist",
        "no dataset found",
        "resource not found",
        "403",
    )
    return any(marker in msg for marker in not_found_markers)


def ensure_kaggle_auth() -> None:
    """Autentica via KaggleHub SDK sem travar execuções não-interativas (batch / CI)."""
    import kagglehub

    has_env_creds = bool(
        os.getenv("KAGGLE_API_TOKEN")
        or (os.getenv("KAGGLE_USERNAME") and os.getenv("KAGGLE_KEY"))
        or (Path.home() / ".kaggle" / "kaggle.json").exists()
        or Path("/root/.config/kaggle/kaggle.json").exists()
    )
    if has_env_creds:
        print("[AUTH] Credenciais do Kaggle detectadas no ambiente.")
        return
    if sys.stdin is not None and sys.stdin.isatty():
        print("Iniciando autenticação manual no Kaggle SDK...")
        kagglehub.login()
    else:
        print(
            "[AUTH] Execução não-interativa detectada; pulando prompt interativo kagglehub.login()."
        )


def bootstrap_dataset(
    handle: str, target_dir: Path, allow_cold_start: bool = False
) -> Path:
    """Se DATASET_PATH estiver vazio, baixa a última versão do dataset do Kaggle ou reconstrói do zero se não existir."""
    target_dir.mkdir(parents=True, exist_ok=True)
    if not is_directory_empty(target_dir):
        print(f"[OK] DATASET_PATH ({target_dir}) já contém dados locais. Pulando download.")
        return target_dir

    print(
        f"[BOOTSTRAP] DATASET_PATH ({target_dir}) está vazio. Baixando '{handle}' via Kaggle SDK..."
    )
    try:
        import kagglehub

        try:
            downloaded_path = Path(
                kagglehub.dataset_download(
                    handle,
                    output_dir=str(target_dir),
                    force_download=True,
                )
            )
        except TypeError as type_err:
            if "output_dir" not in str(type_err):
                raise
            downloaded_path = Path(
                kagglehub.dataset_download(
                    handle,
                    force_download=True,
                )
            )

        if downloaded_path.resolve() != target_dir.resolve() and downloaded_path.exists():
            print(
                f"[SYNC] Copiando arquivos de {downloaded_path} (mount somente-leitura) para {target_dir}..."
            )
            shutil.copytree(downloaded_path, target_dir, dirs_exist_ok=True)
            ensure_writable_tree(target_dir)

        print(f"[OK] Dataset populado com sucesso em {target_dir}.")
    except Exception as exc:
        if allow_cold_start or is_dataset_not_found_error(exc):
            print(
                f"[COLD-START] Dataset '{handle}' não encontrado ou ainda não publicado ({exc}). "
                f"Iniciando reconstrução do zero em {target_dir}."
            )
        else:
            raise RuntimeError(
                f"Falha ao popular DATASET_PATH ({target_dir}) a partir do Kaggle ('{handle}'). "
                "Verifique sua autenticação e conexão de rede."
            ) from exc

    return target_dir


if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    _drive_mount_point = Path("/content/drive")
    if not (_drive_mount_point / "MyDrive").exists():
        try:
            from google.colab import drive

            drive.mount(str(_drive_mount_point))
        except Exception as _exc:
            print(f"[BOOT] Execução headless/CLI detectada ao verificar o Google Drive ({_exc}).")

    _token_cache_file = Path("/root/.config/kaggle/colab_token.env")
    if not os.environ.get("KAGGLE_API_TOKEN"):
        if _token_cache_file.exists():
            os.environ["KAGGLE_API_TOKEN"] = _token_cache_file.read_text(encoding="utf-8").strip()
        else:
            try:
                from google.colab import userdata

                _token = userdata.get("KAGGLE_API_TOKEN")
                if _token:
                    os.environ["KAGGLE_API_TOKEN"] = _token.strip()
                    _token_cache_file.parent.mkdir(parents=True, exist_ok=True)
                    _token_cache_file.write_text(_token.strip(), encoding="utf-8")
                    _token_cache_file.chmod(0o600)
            except Exception as _exc:
                print(
                    f"[BOOT] userdata indisponível no modo CLI ({_exc}); "
                    "utilizando variáveis de ambiente."
                )
    Path(DATASET_PATH).mkdir(parents=True, exist_ok=True)
    Path(STAGING_PATH).mkdir(parents=True, exist_ok=True)
    ensure_kaggle_auth()
    bootstrap_dataset(
        DATASET_HANDLE, Path(DATASET_PATH), allow_cold_start=ALLOW_COLD_START
    )

import gc
import json
import re
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import UTC, datetime

import duckdb
import polars as pl
import pyarrow as pa

# ---------------------------------------------------------------------------
# Configuração de caminhos e parâmetros globais
# ---------------------------------------------------------------------------
ELECTION_YEAR = 2026
GOLD_VERSION = "2.0.0"
MAX_GOLD_FILE_BYTES = int(os.getenv("TSE_MAX_PARQUET_PART_BYTES", str(128 * 1024 * 1024)))
TURNS = [1, 2]
ALL_UFS = [
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA",
    "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN",
    "RO", "RR", "RS", "SC", "SE", "SP", "TO", "ZZ",
]

SILVER_BASE = Path(DATASET_PATH) / "silver"
BRONZE_BASE = Path(DATASET_PATH)
GOLD_OUT = Path(DATASET_PATH) / "gold"
GOLD_OUT.mkdir(parents=True, exist_ok=True)


def find_table_dir(table_name: str, roots=(Path(DATASET_PATH),), max_depth: int = 7) -> Path | None:
    """Localiza <table_name>/ com partições election_year=* sob DATASET_PATH."""
    for root in roots:
        if not root.exists():
            continue
        base = len(root.parts)
        for dirpath, dirnames, _ in os.walk(root):
            p = Path(dirpath)
            if len(p.parts) - base >= max_depth:
                dirnames[:] = []
                continue
            if p.name == table_name and any(d.startswith("election_year=") for d in dirnames):
                return p
    return None


def list_table_parquets(table_name: str, prefer_silver: bool = False) -> list[Path]:
    """Lista de forma ordenada todos os arquivos Parquet de uma tabela sob DATASET_PATH."""
    candidates: list[Path] = []
    if prefer_silver:
        candidates.extend([SILVER_BASE / table_name, Path(DATASET_PATH) / "silver" / table_name])
    else:
        candidates.extend([BRONZE_BASE / table_name, Path(DATASET_PATH) / "bronze" / table_name])
    found_dir = find_table_dir(table_name)
    if found_dir is not None:
        candidates.append(found_dir)
    candidates.append(Path(DATASET_PATH) / table_name)

    for cand in candidates:
        if cand.exists() and cand.is_dir():
            files = sorted(cand.rglob("*.parquet"))
            if files:
                return files
    return []


def write_gold_parquet_staged(df: pl.DataFrame, target_path: Path) -> Path:
    """Grava produto Gold em staging local /tmp e copia após limpeza do destino (seguro para Drive FUSE)."""
    stage_dir = Path(STAGING_PATH) / "gold_analytics"
    stage_dir.mkdir(parents=True, exist_ok=True)
    local_out = stage_dir / target_path.name
    try:
        df.write_parquet(local_out, compression="zstd", compression_level=5)
        file_bytes = local_out.stat().st_size
        if file_bytes > MAX_GOLD_FILE_BYTES:
            raise ValueError(
                f"Arquivo Gold {target_path.name} excedeu o limite de {MAX_GOLD_FILE_BYTES:,} bytes: "
                f"{file_bytes:,} bytes"
            )
        target_path.parent.mkdir(parents=True, exist_ok=True)
        if target_path.exists():
            target_path.unlink()
        shutil.copyfile(local_out, target_path)
        if target_path.stat().st_size != file_bytes:
            raise OSError(f"Tamanho divergente após cópia para {target_path}")
        return target_path
    finally:
        local_out.unlink(missing_ok=True)


def iter_staged_parquets(files: list[Path], cache_subdir: str):
    """Copia arquivos Parquet sequencialmente para o NVMe local com prefetch assíncrono (evita seeks no Drive FUSE)."""
    cache_dir = Path(STAGING_PATH) / cache_subdir
    if cache_dir.exists():
        shutil.rmtree(cache_dir, ignore_errors=True)
    cache_dir.mkdir(parents=True, exist_ok=True)

    def _copy_one(idx: int, src: Path) -> Path:
        dst = cache_dir / f"{idx:05d}_{src.name}"
        shutil.copyfile(src, dst)
        return dst

    try:
        with ThreadPoolExecutor(max_workers=1) as pool:
            next_fut = pool.submit(_copy_one, 0, files[0]) if files else None
            for idx, src in enumerate(files):
                local_path = next_fut.result()
                if idx + 1 < len(files):
                    next_fut = pool.submit(_copy_one, idx + 1, files[idx + 1])
                else:
                    next_fut = None
                try:
                    yield idx, src, local_path
                finally:
                    local_path.unlink(missing_ok=True)
    finally:
        shutil.rmtree(cache_dir, ignore_errors=True)


print("Ambiente Gold configurado com sucesso:")
print(f"  Versão Gold : {GOLD_VERSION}")
print(f"  Polars      : {pl.__version__}")
print(f"  DuckDB      : {duckdb.__version__}")
print(f"  PyArrow     : {pa.__version__}")
print(f"  SILVER_BASE : {SILVER_BASE}")
print(f"  BRONZE_BASE : {BRONZE_BASE}")
print(f"  GOLD_OUT    : {GOLD_OUT}")

[BOOT] userdata indisponível no modo CLI (Requesting secret KAGGLE_API_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.); utilizando variáveis de ambiente.
[AUTH] Execução não-interativa detectada; pulando prompt interativo kagglehub.login().
[OK] DATASET_PATH (/content/drive/MyDrive/tse-eleicoes-dados/data) já contém dados locais. Pulando download.


Ambiente Gold configurado com sucesso:
  Versão Gold : 2.0.0
  Polars      : 2.2.0
  DuckDB      : 2.0.1
  PyArrow     : 26.0.1
  SILVER_BASE : /content/drive/MyDrive/tse-eleicoes-dados/data/silver
  BRONZE_BASE : /content/drive/MyDrive/tse-eleicoes-dados/data
  GOLD_OUT    : /content/drive/MyDrive/tse-eleicoes-dados/data/gold


In [2]:
# ---------------------------------------------------------------------------
# Funções puras de agregação Gold em streaming (escala 4,95 bilhões de linhas)
# ---------------------------------------------------------------------------
EVENT_STREAM_COLS = [
    "election_year",
    "turn",
    "uf",
    "municipio_code",
    "zone_num",
    "section_num",
    "event_code",
    "biometric_attempt_num",
    "classification_status",
    "is_clock_drift",
    "has_timestamp_regression",
]

SESSION_KEY_COLS = [
    "election_year",
    "turn",
    "uf",
    "municipio_code",
    "zone_num",
    "section_num",
]

GOLD_EVENT_DIST_SCHEMA = pl.Schema({
    "event_code": pl.String,
    "turn": pl.Int8,
    "uf": pl.String,
    "event_count": pl.Int64,
    "total_uf_events": pl.Int64,
    "percentage": pl.Float64,
    "clock_drift_count": pl.Int64,
})

GOLD_SESSION_COMP_SCHEMA = pl.Schema({
    "turn": pl.Int8,
    "uf": pl.String,
    "final_state": pl.String,
    "session_count": pl.Int64,
    "total_sessions_uf": pl.Int64,
    "state_share_pct": pl.Float64,
    "completion_rate": pl.Float64,
    "contingency_session_count": pl.Int64,
    "total_voter_cycles": pl.Int64,
    "avg_voter_cycles": pl.Float64,
    "min_voter_cycles": pl.Int32,
    "max_voter_cycles": pl.Int32,
    "total_events": pl.Int64,
})

GOLD_BIO_ATTEMPTS_SCHEMA = pl.Schema({
    "turn": pl.Int8,
    "uf": pl.String,
    "biometric_attempt_num": pl.Int8,
    "attempt_count": pl.Int64,
    "total_attempts_uf": pl.Int64,
    "percentage": pl.Float64,
    "cumulative_count": pl.Int64,
    "cumulative_share": pl.Float64,
})

_SEC_EVENT_SUM_COLS = [
    "total_events",
    "classified_events",
    "unknown_events",
    "voter_enabled_events",
    "biometric_attempt_events",
    "biometric_recognized_events",
    "biometric_not_recognized_events",
    "no_biometric_registered_events",
    "manual_enable_events",
    "inactivity_timeout_events",
    "invalid_key_events",
]


def _reduce_partial_event_dist(frames: list[pl.DataFrame]) -> pl.DataFrame:
    if not frames:
        return pl.DataFrame(
            schema={
                "event_code": pl.String,
                "turn": pl.Int8,
                "uf": pl.String,
                "event_count": pl.Int64,
                "clock_drift_count": pl.Int64,
            }
        )
    return (
        pl.concat(frames, how="vertical")
        .group_by(["event_code", "turn", "uf"])
        .agg(
            pl.col("event_count").cast(pl.Int64).sum().alias("event_count"),
            pl.col("clock_drift_count").cast(pl.Int64).sum().alias("clock_drift_count"),
        )
    )


def _reduce_partial_bio(frames: list[pl.DataFrame]) -> pl.DataFrame:
    if not frames:
        return pl.DataFrame(
            schema={
                "turn": pl.Int8,
                "uf": pl.String,
                "biometric_attempt_num": pl.Int8,
                "attempt_count": pl.Int64,
            }
        )
    return (
        pl.concat(frames, how="vertical")
        .group_by(["turn", "uf", "biometric_attempt_num"])
        .agg(pl.col("attempt_count").cast(pl.Int64).sum().alias("attempt_count"))
    )


def _reduce_partial_section_events(frames: list[pl.DataFrame]) -> pl.DataFrame:
    if not frames:
        schema = {
            "election_year": pl.Int16,
            "turn": pl.Int8,
            "uf": pl.String,
            "municipio_code": pl.Int32,
            "zone_num": pl.Int16,
            "section_num": pl.Int32,
            "has_clock_drift": pl.Boolean,
            "has_timestamp_regression": pl.Boolean,
        }
        for col in _SEC_EVENT_SUM_COLS:
            schema[col] = pl.Int64
        return pl.DataFrame(schema=schema)
    return (
        pl.concat(frames, how="vertical")
        .group_by(SESSION_KEY_COLS)
        .agg(
            pl.col("has_clock_drift").any().alias("has_clock_drift"),
            pl.col("has_timestamp_regression").any().alias("has_timestamp_regression"),
            *[pl.col(c).cast(pl.Int64).sum().alias(c) for c in _SEC_EVENT_SUM_COLS],
        )
    )


def aggregate_urna_events_streaming(
    event_files: list[Path],
    cache_subdir: str = "nb08_event_cache",
    log_every: int = 25,
) -> dict[str, object]:
    """Executa passagem única em streaming sobre todos os shards de fact_urna_event.

    Produz simultaneamente as agregações parciais para gold_event_distribution,
    gold_biometric_attempts e o bloco de eventos de gold_section_turn_summary,
    mantendo o uso de memória limitado independentemente do volume total.
    """
    t0 = time.monotonic()
    dist_partials: list[pl.DataFrame] = []
    bio_partials: list[pl.DataFrame] = []
    sec_partials: list[pl.DataFrame] = []
    total_input_events = 0
    total_bio_rows = 0

    for idx, src, local_path in iter_staged_parquets(event_files, cache_subdir):
        avail_cols = pl.read_parquet_schema(local_path)
        cols_to_read = [c for c in EVENT_STREAM_COLS if c in avail_cols]
        df = pl.read_parquet(local_path, columns=cols_to_read)
        total_input_events += df.height

        add_exprs = []
        if "is_clock_drift" not in df.columns:
            add_exprs.append(pl.lit(False).alias("is_clock_drift"))
        if "has_timestamp_regression" not in df.columns:
            add_exprs.append(pl.lit(False).alias("has_timestamp_regression"))
        if "classification_status" not in df.columns:
            add_exprs.append(pl.lit("CLASSIFIED").alias("classification_status"))
        if add_exprs:
            df = df.with_columns(add_exprs)

        df = df.with_columns(
            pl.col("election_year").cast(pl.Int16),
            pl.col("turn").cast(pl.Int8),
            pl.col("uf").cast(pl.String),
            pl.col("municipio_code").cast(pl.Int32),
            pl.col("zone_num").cast(pl.Int16),
            pl.col("section_num").cast(pl.Int32),
            pl.col("event_code").cast(pl.String),
            pl.col("biometric_attempt_num").cast(pl.Int8),
            pl.col("classification_status").cast(pl.String),
            pl.col("is_clock_drift").fill_null(False).cast(pl.Boolean),
            pl.col("has_timestamp_regression").fill_null(False).cast(pl.Boolean),
        )

        # 1. Parcial gold_event_distribution
        dist_partials.append(
            df.group_by(["event_code", "turn", "uf"]).agg(
                pl.len().cast(pl.Int64).alias("event_count"),
                pl.col("is_clock_drift").cast(pl.Int64).sum().alias("clock_drift_count"),
            )
        )

        # 2. Parcial gold_biometric_attempts
        df_bio = df.filter(
            pl.col("biometric_attempt_num").is_not_null()
            & (pl.col("biometric_attempt_num") > 0)
        )
        total_bio_rows += df_bio.height
        if df_bio.height > 0:
            bio_partials.append(
                df_bio.group_by(["turn", "uf", "biometric_attempt_num"]).agg(
                    pl.len().cast(pl.Int64).alias("attempt_count")
                )
            )
        del df_bio

        # 3. Parcial de eventos por seção × turno
        sec_partials.append(
            df.group_by(SESSION_KEY_COLS).agg(
                pl.col("is_clock_drift").any().alias("has_clock_drift"),
                pl.col("has_timestamp_regression").any().alias("has_timestamp_regression"),
                pl.len().cast(pl.Int64).alias("total_events"),
                (pl.col("classification_status") == "CLASSIFIED").cast(pl.Int64).sum().alias("classified_events"),
                (pl.col("classification_status") != "CLASSIFIED").cast(pl.Int64).sum().alias("unknown_events"),
                (pl.col("event_code") == "VOTER_ENABLED").cast(pl.Int64).sum().alias("voter_enabled_events"),
                (pl.col("event_code") == "BIOMETRIC_READ_ATTEMPT").cast(pl.Int64).sum().alias("biometric_attempt_events"),
                (pl.col("event_code") == "BIOMETRIC_RECOGNIZED").cast(pl.Int64).sum().alias("biometric_recognized_events"),
                (pl.col("event_code") == "BIOMETRIC_NOT_RECOGNIZED").cast(pl.Int64).sum().alias("biometric_not_recognized_events"),
                (pl.col("event_code") == "NO_BIOMETRIC_REGISTERED").cast(pl.Int64).sum().alias("no_biometric_registered_events"),
                (pl.col("event_code") == "MANUAL_ENABLE_PATH").cast(pl.Int64).sum().alias("manual_enable_events"),
                (pl.col("event_code") == "INACTIVITY_TIMEOUT_45S").cast(pl.Int64).sum().alias("inactivity_timeout_events"),
                (pl.col("event_code") == "INVALID_KEY_PRESSED").cast(pl.Int64).sum().alias("invalid_key_events"),
            )
        )
        del df

        if (idx + 1) % 32 == 0:
            dist_partials = [_reduce_partial_event_dist(dist_partials)]
            bio_partials = [_reduce_partial_bio(bio_partials)] if bio_partials else []
            sec_partials = [_reduce_partial_section_events(sec_partials)]
            gc.collect()

        if (idx + 1) % log_every == 0 or (idx + 1) == len(event_files):
            elapsed = time.monotonic() - t0
            print(
                f"  [STREAM fact_urna_event] {idx + 1:>3}/{len(event_files)} shards | "
                f"{total_input_events:>14,} eventos | {elapsed:>6.1f}s"
            )

    return {
        "event_dist_raw": _reduce_partial_event_dist(dist_partials),
        "bio_raw": _reduce_partial_bio(bio_partials),
        "section_events": _reduce_partial_section_events(sec_partials),
        "total_input_events": total_input_events,
        "total_bio_rows": total_bio_rows,
    }


def build_gold_event_distribution(event_dist_raw: pl.DataFrame) -> pl.DataFrame:
    """Finaliza a tabela Gold gold_event_distribution com percentuais por (turn, uf)."""
    if event_dist_raw.is_empty():
        return pl.DataFrame(schema=GOLD_EVENT_DIST_SCHEMA)
    return (
        event_dist_raw
        .with_columns(
            pl.col("event_count").cast(pl.Int64).sum().over(["turn", "uf"]).alias("total_uf_events")
        )
        .with_columns(
            pl.when(pl.col("total_uf_events") > 0)
            .then((pl.col("event_count") * 100.0 / pl.col("total_uf_events")).round(4))
            .otherwise(0.0)
            .alias("percentage")
        )
        .sort(["turn", "uf", "event_count"], descending=[False, False, True])
        .select([
            "event_code", "turn", "uf", "event_count",
            "total_uf_events", "percentage", "clock_drift_count",
        ])
    )


def build_gold_biometric_attempts(bio_raw: pl.DataFrame) -> pl.DataFrame:
    """Finaliza a tabela Gold gold_biometric_attempts com curva cumulativa por (turn, uf)."""
    if bio_raw.is_empty():
        return pl.DataFrame(schema=GOLD_BIO_ATTEMPTS_SCHEMA)
    return (
        bio_raw
        .sort(["turn", "uf", "biometric_attempt_num"])
        .with_columns(
            pl.col("attempt_count").cast(pl.Int64).sum().over(["turn", "uf"]).alias("total_attempts_uf"),
            pl.col("attempt_count").cast(pl.Int64).cum_sum().over(["turn", "uf"]).alias("cumulative_count"),
        )
        .with_columns(
            pl.when(pl.col("total_attempts_uf") > 0)
            .then((pl.col("attempt_count") * 100.0 / pl.col("total_attempts_uf")).round(4))
            .otherwise(0.0)
            .alias("percentage"),
            pl.when(pl.col("total_attempts_uf") > 0)
            .then((pl.col("cumulative_count") * 100.0 / pl.col("total_attempts_uf")).round(4))
            .otherwise(0.0)
            .alias("cumulative_share"),
        )
        .select([
            "turn", "uf", "biometric_attempt_num", "attempt_count",
            "total_attempts_uf", "percentage", "cumulative_count", "cumulative_share",
        ])
    )


def build_gold_session_completeness(df_sessions: pl.DataFrame) -> pl.DataFrame:
    """Constrói gold_session_completeness com soma segura em Int64 (sem overflow de Int32)."""
    if df_sessions.is_empty():
        return pl.DataFrame(schema=GOLD_SESSION_COMP_SCHEMA)
    return (
        df_sessions
        .group_by(["turn", "uf", "final_state"])
        .agg(
            pl.len().cast(pl.Int64).alias("session_count"),
            pl.col("has_contingency").fill_null(False).cast(pl.Int64).sum().alias("contingency_session_count"),
            pl.col("voter_cycles").cast(pl.Int64).sum().alias("total_voter_cycles"),
            pl.col("voter_cycles").cast(pl.Float64).mean().round(2).alias("avg_voter_cycles"),
            pl.col("voter_cycles").min().cast(pl.Int32).alias("min_voter_cycles"),
            pl.col("voter_cycles").max().cast(pl.Int32).alias("max_voter_cycles"),
            pl.col("event_count").cast(pl.Int64).sum().alias("total_events"),
        )
        .with_columns(
            pl.col("session_count").sum().over(["turn", "uf"]).alias("total_sessions_uf"),
            pl.when(pl.col("final_state") == "CLOSED")
            .then(pl.col("session_count"))
            .otherwise(0)
            .sum()
            .over(["turn", "uf"])
            .alias("_closed_count_uf"),
        )
        .with_columns(
            pl.when(pl.col("total_sessions_uf") > 0)
            .then((pl.col("session_count") * 100.0 / pl.col("total_sessions_uf")).round(4))
            .otherwise(0.0)
            .alias("state_share_pct"),
            pl.when(pl.col("total_sessions_uf") > 0)
            .then((pl.col("_closed_count_uf") * 100.0 / pl.col("total_sessions_uf")).round(4))
            .otherwise(0.0)
            .alias("completion_rate"),
        )
        .drop("_closed_count_uf")
        .sort(["turn", "uf", "final_state"])
        .select([
            "turn", "uf", "final_state", "session_count", "total_sessions_uf",
            "state_share_pct", "completion_rate", "contingency_session_count",
            "total_voter_cycles", "avg_voter_cycles", "min_voter_cycles",
            "max_voter_cycles", "total_events",
        ])
    )


def _extract_turn_from_path(path: Path) -> int | None:
    m = re.search(r"turn=(\d+)", str(path))
    return int(m.group(1)) if m else None


def load_bronze_bu_turnout(files: list[Path]) -> pl.DataFrame:
    """Extrai comparecimento oficial por seção × turno de fact_boletim_urna (CD_CARGO_PERGUNTA = 1 — Presidente)."""
    empty = pl.DataFrame(
        schema={
            "turn": pl.Int8,
            "uf": pl.String,
            "municipio_code": pl.Int32,
            "zone_num": pl.Int16,
            "section_num": pl.Int32,
            "bu_municipio_name": pl.String,
            "bu_local_votacao_num": pl.Int32,
            "qt_aptos": pl.Int64,
            "qt_comparecimento": pl.Int64,
            "qt_abstencoes": pl.Int64,
            "qt_elei_biom_sem_habilitacao": pl.Int64,
        }
    )
    if not files:
        return empty
    cols = [
        "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NM_MUNICIPIO", "NR_ZONA", "NR_SECAO",
        "NR_LOCAL_VOTACAO", "CD_CARGO_PERGUNTA", "QT_APTOS", "QT_COMPARECIMENTO",
        "QT_ABSTENCOES", "QT_ELEI_BIOM_SEM_HABILITACAO",
    ]
    parts: list[pl.DataFrame] = []
    for _idx, _src, local_path in iter_staged_parquets(files, "nb08_bu_cache"):
        df = pl.read_parquet(local_path, columns=cols)
        df_pres = df.filter(pl.col("CD_CARGO_PERGUNTA") == 1)
        if df_pres.height == 0:
            df_pres = df
        parts.append(
            df_pres.group_by(["NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO"]).agg(
                pl.col("NM_MUNICIPIO").drop_nulls().first().cast(pl.String).alias("bu_municipio_name"),
                pl.col("NR_LOCAL_VOTACAO").drop_nulls().first().cast(pl.Int32).alias("bu_local_votacao_num"),
                pl.col("QT_APTOS").cast(pl.Int64).max().alias("qt_aptos"),
                pl.col("QT_COMPARECIMENTO").cast(pl.Int64).max().alias("qt_comparecimento"),
                pl.col("QT_ABSTENCOES").cast(pl.Int64).max().alias("qt_abstencoes"),
                pl.col("QT_ELEI_BIOM_SEM_HABILITACAO").cast(pl.Int64).max().alias("qt_elei_biom_sem_habilitacao"),
            ).select(
                pl.col("NR_TURNO").cast(pl.Int8).alias("turn"),
                pl.col("SG_UF").cast(pl.String).alias("uf"),
                pl.col("CD_MUNICIPIO").cast(pl.Int32).alias("municipio_code"),
                pl.col("NR_ZONA").cast(pl.Int16).alias("zone_num"),
                pl.col("NR_SECAO").cast(pl.Int32).alias("section_num"),
                "bu_municipio_name",
                "bu_local_votacao_num",
                "qt_aptos",
                "qt_comparecimento",
                "qt_abstencoes",
                "qt_elei_biom_sem_habilitacao",
            )
        )
        del df, df_pres
    if not parts:
        return empty
    return (
        pl.concat(parts, how="vertical")
        .group_by(["turn", "uf", "municipio_code", "zone_num", "section_num"])
        .agg(
            pl.col("bu_municipio_name").drop_nulls().first().alias("bu_municipio_name"),
            pl.col("bu_local_votacao_num").drop_nulls().first().alias("bu_local_votacao_num"),
            pl.col("qt_aptos").max().alias("qt_aptos"),
            pl.col("qt_comparecimento").max().alias("qt_comparecimento"),
            pl.col("qt_abstencoes").max().alias("qt_abstencoes"),
            pl.col("qt_elei_biom_sem_habilitacao").max().alias("qt_elei_biom_sem_habilitacao"),
        )
    )


def load_bronze_local_votacao(files: list[Path]) -> pl.DataFrame:
    """Extrai nome e coordenadas geográficas do local de votação a partir de dim_local_votacao_secao."""
    empty = pl.DataFrame(
        schema={
            "turn": pl.Int8,
            "uf": pl.String,
            "municipio_code": pl.Int32,
            "zone_num": pl.Int16,
            "section_num": pl.Int32,
            "loc_municipio_name": pl.String,
            "loc_local_votacao_num": pl.Int32,
            "local_votacao_name": pl.String,
            "latitude": pl.Float64,
            "longitude": pl.Float64,
        }
    )
    if not files:
        return empty
    cols = [
        "NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NM_MUNICIPIO", "NR_ZONA", "NR_SECAO",
        "NR_LOCAL_VOTACAO", "NM_LOCAL_VOTACAO", "NR_LATITUDE", "NR_LONGITUDE",
    ]
    parts: list[pl.DataFrame] = []
    for _idx, _src, local_path in iter_staged_parquets(files, "nb08_loc_cache"):
        df = pl.read_parquet(local_path, columns=cols)
        lat_num = (
            pl.col("NR_LATITUDE").cast(pl.String).str.strip_chars().str.replace(",", ".")
            .cast(pl.Float64, strict=False)
        )
        lon_num = (
            pl.col("NR_LONGITUDE").cast(pl.String).str.strip_chars().str.replace(",", ".")
            .cast(pl.Float64, strict=False)
        )
        parts.append(
            df.select(
                pl.col("NR_TURNO").cast(pl.Int8).alias("turn"),
                pl.col("SG_UF").cast(pl.String).alias("uf"),
                pl.col("CD_MUNICIPIO").cast(pl.Int32).alias("municipio_code"),
                pl.col("NR_ZONA").cast(pl.Int16).alias("zone_num"),
                pl.col("NR_SECAO").cast(pl.Int32).alias("section_num"),
                pl.col("NM_MUNICIPIO").cast(pl.String).alias("loc_municipio_name"),
                pl.col("NR_LOCAL_VOTACAO").cast(pl.Int32).alias("loc_local_votacao_num"),
                pl.col("NM_LOCAL_VOTACAO").cast(pl.String).alias("local_votacao_name"),
                pl.when(lat_num.is_between(-90.0, 90.0) & (lat_num != -1.0))
                .then(lat_num)
                .otherwise(None)
                .alias("latitude"),
                pl.when(lon_num.is_between(-180.0, 180.0) & (lon_num != -1.0))
                .then(lon_num)
                .otherwise(None)
                .alias("longitude"),
            )
            .unique(subset=["turn", "uf", "municipio_code", "zone_num", "section_num"], keep="first")
        )
        del df
    if not parts:
        return empty
    return (
        pl.concat(parts, how="vertical")
        .unique(subset=["turn", "uf", "municipio_code", "zone_num", "section_num"], keep="first")
    )


def load_bronze_eleitorado_perfil(files: list[Path]) -> pl.DataFrame:
    """Agrega perfil cadastral de biometria e deficiência por seção a partir de fact_eleitorado_perfil_secao."""
    empty = pl.DataFrame(
        schema={
            "uf": pl.String,
            "municipio_code": pl.Int32,
            "zone_num": pl.Int16,
            "section_num": pl.Int32,
            "qt_eleitores_perfil": pl.Int64,
            "qt_eleitores_biometria": pl.Int64,
            "qt_eleitores_deficiencia": pl.Int64,
        }
    )
    if not files:
        return empty
    cols = [
        "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO",
        "QT_ELEITORES", "QT_ELEITORES_BIOMETRIA", "QT_ELEITORES_DEFICIENCIA",
    ]
    parts: list[pl.DataFrame] = []
    for _idx, _src, local_path in iter_staged_parquets(files, "nb08_perfil_cache"):
        df = pl.read_parquet(local_path, columns=cols)
        parts.append(
            df.group_by(["SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO"]).agg(
                pl.col("QT_ELEITORES").cast(pl.Int64).sum().alias("qt_eleitores_perfil"),
                pl.col("QT_ELEITORES_BIOMETRIA").cast(pl.Int64).sum().alias("qt_eleitores_biometria"),
                pl.col("QT_ELEITORES_DEFICIENCIA").cast(pl.Int64).sum().alias("qt_eleitores_deficiencia"),
            ).select(
                pl.col("SG_UF").cast(pl.String).alias("uf"),
                pl.col("CD_MUNICIPIO").cast(pl.Int32).alias("municipio_code"),
                pl.col("NR_ZONA").cast(pl.Int16).alias("zone_num"),
                pl.col("NR_SECAO").cast(pl.Int32).alias("section_num"),
                "qt_eleitores_perfil",
                "qt_eleitores_biometria",
                "qt_eleitores_deficiencia",
            )
        )
        del df
    if not parts:
        return empty
    return (
        pl.concat(parts, how="vertical")
        .group_by(["uf", "municipio_code", "zone_num", "section_num"])
        .agg(
            pl.col("qt_eleitores_perfil").sum().alias("qt_eleitores_perfil"),
            pl.col("qt_eleitores_biometria").sum().alias("qt_eleitores_biometria"),
            pl.col("qt_eleitores_deficiencia").sum().alias("qt_eleitores_deficiencia"),
        )
    )


def load_bronze_correspondencia(files: list[Path]) -> pl.DataFrame:
    """Extrai sinalização de troca de urna e divergência a partir de bridge_correspondencia_efetivada."""
    empty = pl.DataFrame(
        schema={
            "turn": pl.Int8,
            "uf": pl.String,
            "municipio_code": pl.Int32,
            "zone_num": pl.Int16,
            "section_num": pl.Int32,
            "corresp_urna_swapped": pl.Boolean,
            "corresp_divergence_type": pl.String,
        }
    )
    if not files:
        return empty
    cols = [
        "CD_PLEITO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO",
        "NR_URNA_ESPERADA", "NR_URNA_EFETIVADA", "TP_DIVERGENCIA",
    ]
    parts: list[pl.DataFrame] = []
    for _idx, src, local_path in iter_staged_parquets(files, "nb08_corresp_cache"):
        path_turn = _extract_turn_from_path(src)
        df = pl.read_parquet(local_path, columns=cols)
        if path_turn is not None:
            turn_expr = pl.lit(path_turn, dtype=pl.Int8).alias("turn")
        else:
            turn_expr = (
                pl.when(pl.col("CD_PLEITO") == 3220)
                .then(pl.lit(1, dtype=pl.Int8))
                .when(pl.col("CD_PLEITO") == 3221)
                .then(pl.lit(2, dtype=pl.Int8))
                .otherwise(pl.lit(1, dtype=pl.Int8))
                .alias("turn")
            )
        div_clean = (
            pl.when(
                pl.col("TP_DIVERGENCIA").is_not_null()
                & (pl.col("TP_DIVERGENCIA").cast(pl.String).str.strip_chars() != "")
                & (pl.col("TP_DIVERGENCIA").cast(pl.String).str.strip_chars() != "#NULO#")
            )
            .then(pl.col("TP_DIVERGENCIA").cast(pl.String).str.strip_chars())
            .otherwise(None)
            .alias("_div_clean")
        )
        swapped_expr = (
            pl.col("NR_URNA_ESPERADA").is_not_null()
            & pl.col("NR_URNA_EFETIVADA").is_not_null()
            & (pl.col("NR_URNA_ESPERADA") != pl.col("NR_URNA_EFETIVADA"))
        ).alias("_swapped")
        parts.append(
            df.with_columns(turn_expr, div_clean, swapped_expr)
            .group_by(["turn", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO"])
            .agg(
                pl.col("_swapped").any().alias("corresp_urna_swapped"),
                pl.col("_div_clean").drop_nulls().first().alias("corresp_divergence_type"),
            )
            .select(
                "turn",
                pl.col("SG_UF").cast(pl.String).alias("uf"),
                pl.col("CD_MUNICIPIO").cast(pl.Int32).alias("municipio_code"),
                pl.col("NR_ZONA").cast(pl.Int16).alias("zone_num"),
                pl.col("NR_SECAO").cast(pl.Int32).alias("section_num"),
                "corresp_urna_swapped",
                "corresp_divergence_type",
            )
        )
        del df
    if not parts:
        return empty
    return (
        pl.concat(parts, how="vertical")
        .group_by(["turn", "uf", "municipio_code", "zone_num", "section_num"])
        .agg(
            pl.col("corresp_urna_swapped").any().alias("corresp_urna_swapped"),
            pl.col("corresp_divergence_type").drop_nulls().first().alias("corresp_divergence_type"),
        )
    )


def load_bronze_urna_model(files: list[Path]) -> pl.DataFrame:
    """Extrai modelo de hardware da urna (DS_MODELO_URNA) por seção a partir de fact_detalhe_votacao_secao."""
    empty = pl.DataFrame(
        schema={
            "uf": pl.String,
            "municipio_code": pl.Int32,
            "zone_num": pl.Int16,
            "section_num": pl.Int32,
            "urna_model_desc": pl.String,
        }
    )
    if not files:
        return empty
    cols = ["SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO", "DS_MODELO_URNA"]
    parts: list[pl.DataFrame] = []
    for _idx, _src, local_path in iter_staged_parquets(files, "nb08_modelo_cache"):
        df = pl.read_parquet(local_path, columns=cols)
        parts.append(
            df.filter(
                pl.col("DS_MODELO_URNA").is_not_null()
                & (pl.col("DS_MODELO_URNA").cast(pl.String).str.strip_chars() != "")
                & (pl.col("DS_MODELO_URNA").cast(pl.String).str.strip_chars() != "#NULO#")
            )
            .select(
                pl.col("SG_UF").cast(pl.String).alias("uf"),
                pl.col("CD_MUNICIPIO").cast(pl.Int32).alias("municipio_code"),
                pl.col("NR_ZONA").cast(pl.Int16).alias("zone_num"),
                pl.col("NR_SECAO").cast(pl.Int32).alias("section_num"),
                pl.col("DS_MODELO_URNA").cast(pl.String).str.strip_chars().alias("urna_model_desc"),
            )
            .unique(subset=["uf", "municipio_code", "zone_num", "section_num"], keep="first")
        )
        del df
    if not parts:
        return empty
    return (
        pl.concat(parts, how="vertical")
        .unique(subset=["uf", "municipio_code", "zone_num", "section_num"], keep="first")
    )


def build_gold_section_turn_summary(
    df_sessions: pl.DataFrame,
    df_section_events: pl.DataFrame,
    df_bu: pl.DataFrame,
    df_loc: pl.DataFrame,
    df_perfil: pl.DataFrame,
    df_corresp: pl.DataFrame,
    df_modelo: pl.DataFrame,
) -> pl.DataFrame:
    """Constrói gold_section_turn_summary no grão section_turn (1 linha por seção × turno)."""
    sec_sessions = (
        df_sessions
        .group_by(SESSION_KEY_COLS)
        .agg(
            pl.col("urna_serial").n_unique().cast(pl.Int32).alias("urna_count"),
            pl.col("is_complete").all().alias("all_sessions_complete"),
            pl.col("has_contingency").any().alias("has_contingency"),
            pl.col("first_event_ts").min().alias("first_event_ts"),
            pl.col("last_event_ts").max().alias("last_event_ts"),
            pl.col("voter_cycles").cast(pl.Int64).sum().alias("voter_cycles"),
            pl.col("event_count").cast(pl.Int64).sum().alias("_session_event_count"),
        )
        .with_columns(
            pl.when(pl.col("all_sessions_complete"))
            .then(pl.lit("CLOSED"))
            .otherwise(pl.lit("INCOMPLETE"))
            .alias("session_final_state")
        )
    )

    df = sec_sessions.join(df_section_events, on=SESSION_KEY_COLS, how="left")
    df = (
        df
        .join(df_bu, on=["turn", "uf", "municipio_code", "zone_num", "section_num"], how="left")
        .join(df_loc, on=["turn", "uf", "municipio_code", "zone_num", "section_num"], how="left")
        .join(df_perfil, on=["uf", "municipio_code", "zone_num", "section_num"], how="left")
        .join(df_corresp, on=["turn", "uf", "municipio_code", "zone_num", "section_num"], how="left")
        .join(df_modelo, on=["uf", "municipio_code", "zone_num", "section_num"], how="left")
    )

    df = df.with_columns(
        pl.coalesce([pl.col("bu_municipio_name"), pl.col("loc_municipio_name")]).alias("municipio_name"),
        pl.coalesce([pl.col("loc_local_votacao_num"), pl.col("bu_local_votacao_num")]).alias("local_votacao_num"),
        pl.col("has_clock_drift").fill_null(False).alias("has_clock_drift"),
        pl.col("has_timestamp_regression").fill_null(False).alias("has_timestamp_regression"),
        pl.coalesce([pl.col("total_events"), pl.col("_session_event_count")]).cast(pl.Int64).alias("total_events"),
        *[pl.col(c).fill_null(0).cast(pl.Int64).alias(c) for c in _SEC_EVENT_SUM_COLS if c != "total_events"],
    )

    duration_expr = (
        pl.when(
            (~pl.col("has_clock_drift"))
            & pl.col("first_event_ts").is_not_null()
            & pl.col("last_event_ts").is_not_null()
            & (pl.col("last_event_ts") >= pl.col("first_event_ts"))
        )
        .then((pl.col("last_event_ts") - pl.col("first_event_ts")).dt.total_seconds().cast(pl.Int64))
        .otherwise(pl.lit(None, dtype=pl.Int64))
        .alias("observed_duration_sec")
    )

    bio_denom = pl.col("biometric_recognized_events") + pl.col("biometric_not_recognized_events")
    df = df.with_columns(
        duration_expr,
        pl.when(bio_denom > 0)
        .then((pl.col("biometric_recognized_events") * 100.0 / bio_denom).round(4).clip(0.0, 100.0))
        .otherwise(pl.lit(None, dtype=pl.Float64))
        .alias("biometric_recognition_rate_pct"),
        pl.when(pl.col("voter_cycles") > 0)
        .then((pl.col("manual_enable_events") * 100.0 / pl.col("voter_cycles")).round(4).clip(0.0, 100.0))
        .otherwise(pl.lit(None, dtype=pl.Float64))
        .alias("manual_enable_rate_pct"),
        pl.when(pl.col("qt_aptos").is_not_null() & (pl.col("qt_aptos") > 0))
        .then((pl.col("qt_comparecimento") * 100.0 / pl.col("qt_aptos")).round(4).clip(0.0, 100.0))
        .otherwise(pl.lit(None, dtype=pl.Float64))
        .alias("turnout_rate_pct"),
        pl.when(pl.col("qt_eleitores_perfil").is_not_null() & (pl.col("qt_eleitores_perfil") > 0))
        .then((pl.col("qt_eleitores_biometria") * 100.0 / pl.col("qt_eleitores_perfil")).round(4).clip(0.0, 100.0))
        .otherwise(pl.lit(None, dtype=pl.Float64))
        .alias("biometria_coverage_pct"),
    )

    return (
        df
        .sort(SESSION_KEY_COLS)
        .select([
            "election_year", "turn", "uf", "municipio_code", "municipio_name",
            "zone_num", "section_num", "local_votacao_num", "local_votacao_name",
            "latitude", "longitude",
            "urna_count", "session_final_state", "all_sessions_complete",
            "has_contingency", "has_clock_drift", "has_timestamp_regression",
            "first_event_ts", "last_event_ts", "observed_duration_sec",
            "total_events", "classified_events", "unknown_events",
            "voter_enabled_events", "voter_cycles",
            "biometric_attempt_events", "biometric_recognized_events",
            "biometric_not_recognized_events", "no_biometric_registered_events",
            "manual_enable_events", "inactivity_timeout_events", "invalid_key_events",
            "biometric_recognition_rate_pct", "manual_enable_rate_pct",
            "qt_aptos", "qt_comparecimento", "qt_abstencoes",
            "qt_elei_biom_sem_habilitacao", "turnout_rate_pct", "urna_model_desc",
            "qt_eleitores_perfil", "qt_eleitores_biometria",
            "biometria_coverage_pct", "qt_eleitores_deficiencia",
            "corresp_urna_swapped", "corresp_divergence_type",
        ])
    )


def build_gold_municipality_turn_summary(df_sec: pl.DataFrame) -> pl.DataFrame:
    """Consolida gold_section_turn_summary no grão município × turno."""
    return (
        df_sec
        .group_by(["election_year", "turn", "uf", "municipio_code"])
        .agg(
            pl.col("municipio_name").drop_nulls().first().alias("municipio_name"),
            pl.len().cast(pl.Int64).alias("section_count"),
            (pl.col("session_final_state") == "CLOSED").cast(pl.Int64).sum().alias("closed_section_count"),
            pl.col("corresp_urna_swapped").fill_null(False).cast(pl.Int64).sum().alias("urna_swapped_count"),
            pl.col("has_clock_drift").fill_null(False).cast(pl.Int64).sum().alias("clock_drift_section_count"),
            pl.col("qt_aptos").cast(pl.Int64).sum().alias("qt_aptos"),
            pl.col("qt_comparecimento").cast(pl.Int64).sum().alias("qt_comparecimento"),
            pl.col("qt_abstencoes").cast(pl.Int64).sum().alias("qt_abstencoes"),
            pl.col("voter_cycles").cast(pl.Int64).sum().alias("voter_cycles"),
            pl.col("total_events").cast(pl.Int64).sum().alias("total_events"),
            pl.col("biometric_attempt_events").cast(pl.Int64).sum().alias("biometric_attempt_events"),
            pl.col("biometric_recognized_events").cast(pl.Int64).sum().alias("biometric_recognized_events"),
            pl.col("biometric_not_recognized_events").cast(pl.Int64).sum().alias("_bio_not_rec"),
            pl.col("manual_enable_events").cast(pl.Int64).sum().alias("manual_enable_events"),
            pl.col("inactivity_timeout_events").cast(pl.Int64).sum().alias("inactivity_timeout_events"),
            pl.col("invalid_key_events").cast(pl.Int64).sum().alias("invalid_key_events"),
            pl.col("observed_duration_sec").cast(pl.Float64).mean().round(2).alias("avg_observed_duration_sec"),
            pl.col("qt_eleitores_perfil").cast(pl.Int64).sum().alias("qt_eleitores_perfil"),
            pl.col("qt_eleitores_biometria").cast(pl.Int64).sum().alias("qt_eleitores_biometria"),
            pl.col("qt_eleitores_deficiencia").cast(pl.Int64).sum().alias("qt_eleitores_deficiencia"),
        )
        .with_columns(
            pl.when(pl.col("qt_aptos") > 0)
            .then((pl.col("qt_comparecimento") * 100.0 / pl.col("qt_aptos")).round(4).clip(0.0, 100.0))
            .otherwise(pl.lit(None, dtype=pl.Float64))
            .alias("turnout_rate_pct"),
            pl.when((pl.col("biometric_recognized_events") + pl.col("_bio_not_rec")) > 0)
            .then(
                (
                    pl.col("biometric_recognized_events")
                    * 100.0
                    / (pl.col("biometric_recognized_events") + pl.col("_bio_not_rec"))
                )
                .round(4)
                .clip(0.0, 100.0)
            )
            .otherwise(pl.lit(None, dtype=pl.Float64))
            .alias("biometric_recognition_rate_pct"),
            pl.when(pl.col("voter_cycles") > 0)
            .then((pl.col("manual_enable_events") * 100.0 / pl.col("voter_cycles")).round(4).clip(0.0, 100.0))
            .otherwise(pl.lit(None, dtype=pl.Float64))
            .alias("manual_enable_rate_pct"),
            pl.when(pl.col("qt_eleitores_perfil") > 0)
            .then((pl.col("qt_eleitores_biometria") * 100.0 / pl.col("qt_eleitores_perfil")).round(4).clip(0.0, 100.0))
            .otherwise(pl.lit(None, dtype=pl.Float64))
            .alias("biometria_coverage_pct"),
        )
        .drop("_bio_not_rec")
        .sort(["election_year", "turn", "uf", "municipio_code"])
        .select([
            "election_year", "turn", "uf", "municipio_code", "municipio_name",
            "section_count", "closed_section_count", "urna_swapped_count",
            "clock_drift_section_count",
            "qt_aptos", "qt_comparecimento", "qt_abstencoes", "turnout_rate_pct",
            "voter_cycles", "total_events",
            "biometric_attempt_events", "biometric_recognized_events",
            "manual_enable_events", "inactivity_timeout_events", "invalid_key_events",
            "biometric_recognition_rate_pct", "manual_enable_rate_pct",
            "avg_observed_duration_sec",
            "qt_eleitores_perfil", "qt_eleitores_biometria",
            "biometria_coverage_pct", "qt_eleitores_deficiencia",
        ])
    )


print("Funções de agregação Gold em streaming registradas com sucesso.")

Funções de agregação Gold em streaming registradas com sucesso.


In [3]:
# ---------------------------------------------------------------------------
# Execução em Streaming sobre Silver (fact_urna_event e fact_voting_session)
# ---------------------------------------------------------------------------
event_files = list_table_parquets("fact_urna_event", prefer_silver=True)
session_files = list_table_parquets("fact_voting_session", prefer_silver=True)

if not event_files:
    raise FileNotFoundError(
        "Nenhum Parquet de silver/fact_urna_event localizado. Execute o NB-05 antes da camada Gold."
    )
if not session_files:
    raise FileNotFoundError(
        "Nenhum Parquet de silver/fact_voting_session localizado. Execute o NB-06 antes da camada Gold."
    )

print("Arquivos Silver localizados:")
print(f"  fact_urna_event     : {len(event_files)} shards")
print(f"  fact_voting_session : {len(session_files)} partições")

session_parts: list[pl.DataFrame] = []
for _idx, _src, local_path in iter_staged_parquets(session_files, "nb08_session_cache"):
    session_parts.append(pl.read_parquet(local_path))
df_sessions = pl.concat(session_parts, how="vertical")
del session_parts
print(f"[OK] fact_voting_session carregada: {df_sessions.height:,} sessões.")

stream_res = aggregate_urna_events_streaming(event_files)
total_input_events = int(stream_res["total_input_events"])
total_bio_rows = int(stream_res["total_bio_rows"])

if total_input_events == 0 or df_sessions.height == 0:
    raise RuntimeError("Camada Silver vazia: Gold não pode ser materializada sem registros.")

gold_event_dist = build_gold_event_distribution(stream_res["event_dist_raw"])
event_dist_path = write_gold_parquet_staged(gold_event_dist, GOLD_OUT / "gold_event_distribution.parquet")
print(f"[gold_event_distribution] {gold_event_dist.height:,} linhas -> {event_dist_path.name} ({event_dist_path.stat().st_size:,} bytes)")

gold_session_comp = build_gold_session_completeness(df_sessions)
session_comp_path = write_gold_parquet_staged(gold_session_comp, GOLD_OUT / "gold_session_completeness.parquet")
print(f"[gold_session_completeness] {gold_session_comp.height:,} linhas -> {session_comp_path.name} ({session_comp_path.stat().st_size:,} bytes)")

gold_bio_attempts = build_gold_biometric_attempts(stream_res["bio_raw"])
bio_attempts_path = write_gold_parquet_staged(gold_bio_attempts, GOLD_OUT / "gold_biometric_attempts.parquet")
print(f"[gold_biometric_attempts] {gold_bio_attempts.height:,} linhas -> {bio_attempts_path.name} ({bio_attempts_path.stat().st_size:,} bytes)")

Arquivos Silver localizados:
  fact_urna_event     : 322 shards
  fact_voting_session : 56 partições
[OK] fact_voting_session carregada: 998,355 sessões.


  [STREAM fact_urna_event]  25/322 shards |    646,586,144 eventos |   80.1s


  [STREAM fact_urna_event]  50/322 shards |  1,282,302,779 eventos |  151.3s


  [STREAM fact_urna_event]  75/322 shards |  1,941,049,679 eventos |  223.6s


  [STREAM fact_urna_event] 100/322 shards |  2,578,035,201 eventos |  294.4s


  [STREAM fact_urna_event] 125/322 shards |  3,061,155,425 eventos |  345.7s


... (5 outputs left out)

  [STREAM fact_urna_event] 275/322 shards |  4,508,895,775 eventos |  495.3s


  [STREAM fact_urna_event] 300/322 shards |  4,752,296,008 eventos |  521.7s


  [STREAM fact_urna_event] 322/322 shards |  4,954,541,320 eventos |  540.9s


[gold_event_distribution] 832 linhas -> gold_event_distribution.parquet (10,855 bytes)
[gold_session_completeness] 57 linhas -> gold_session_completeness.parquet (6,479 bytes)
[gold_biometric_attempts] 216 linhas -> gold_biometric_attempts.parquet (7,854 bytes)


In [4]:
# ---------------------------------------------------------------------------
# Produtos Gold Enriquecidos: gold_section_turn_summary & gold_municipality_turn_summary
# ---------------------------------------------------------------------------
t_enr = time.monotonic()

bu_files = list_table_parquets("fact_boletim_urna", prefer_silver=False)
loc_files = list_table_parquets("dim_local_votacao_secao", prefer_silver=False)
perfil_files = list_table_parquets("fact_eleitorado_perfil_secao", prefer_silver=False)
corresp_files = list_table_parquets("bridge_correspondencia_efetivada", prefer_silver=False)
detalhe_files = list_table_parquets("fact_detalhe_votacao_secao", prefer_silver=False)

print("Carregando atributos oficiais Bronze para enriquecimento (staging NVMe sequencial)...")
df_bu = load_bronze_bu_turnout(bu_files)
print(f"  [OK] fact_boletim_urna (Presidente)    : {df_bu.height:,} seções × turno ({len(bu_files)} arquivos)")
df_loc = load_bronze_local_votacao(loc_files)
print(f"  [OK] dim_local_votacao_secao           : {df_loc.height:,} seções × turno ({len(loc_files)} arquivos)")
df_perfil = load_bronze_eleitorado_perfil(perfil_files)
print(f"  [OK] fact_eleitorado_perfil_secao      : {df_perfil.height:,} seções ({len(perfil_files)} arquivos)")
df_corresp = load_bronze_correspondencia(corresp_files)
print(f"  [OK] bridge_correspondencia_efetivada  : {df_corresp.height:,} seções × turno ({len(corresp_files)} arquivos)")
df_modelo = load_bronze_urna_model(detalhe_files)
print(f"  [OK] fact_detalhe_votacao_secao modelo : {df_modelo.height:,} seções ({len(detalhe_files)} arquivos)")

gold_section_summary = build_gold_section_turn_summary(
    df_sessions=df_sessions,
    df_section_events=stream_res["section_events"],
    df_bu=df_bu,
    df_loc=df_loc,
    df_perfil=df_perfil,
    df_corresp=df_corresp,
    df_modelo=df_modelo,
)
section_summary_path = write_gold_parquet_staged(
    gold_section_summary, GOLD_OUT / "gold_section_turn_summary.parquet"
)
print(
    f"[gold_section_turn_summary] {gold_section_summary.height:,} linhas -> "
    f"{section_summary_path.name} ({section_summary_path.stat().st_size / (1024**2):.2f} MB)"
)

gold_mun_summary = build_gold_municipality_turn_summary(gold_section_summary)
mun_summary_path = write_gold_parquet_staged(
    gold_mun_summary, GOLD_OUT / "gold_municipality_turn_summary.parquet"
)
print(
    f"[gold_municipality_turn_summary] {gold_mun_summary.height:,} linhas -> "
    f"{mun_summary_path.name} ({mun_summary_path.stat().st_size / (1024**2):.2f} MB)"
)
print(f"Enriquecimento Gold concluído em {time.monotonic() - t_enr:.1f}s.")

Carregando atributos oficiais Bronze para enriquecimento (staging NVMe sequencial)...


  [OK] fact_boletim_urna (Presidente)    : 998,411 seções × turno (56 arquivos)


  [OK] dim_local_votacao_secao           : 1,034,358 seções × turno (56 arquivos)


  [OK] fact_eleitorado_perfil_secao      : 516,785 seções (28 arquivos)


  [OK] bridge_correspondencia_efetivada  : 998,414 seções × turno (56 arquivos)


  [OK] fact_detalhe_votacao_secao modelo : 497,897 seções (33 arquivos)


[gold_section_turn_summary] 998,354 linhas -> gold_section_turn_summary.parquet (46.87 MB)
[gold_municipality_turn_summary] 11,376 linhas -> gold_municipality_turn_summary.parquet (0.61 MB)
Enriquecimento Gold concluído em 184.1s.


In [5]:
# ---------------------------------------------------------------------------
# Validações, Asserções Quantitativas e Reconciliação Independente com DuckDB
# ---------------------------------------------------------------------------
print("Iniciando bateria de asserções e testes de consistência Gold (5 tabelas)...")

reconciled_events = int(gold_event_dist["event_count"].cast(pl.Int64).sum())
assert reconciled_events == total_input_events, (
    f"Divergência de contagem de eventos: soma na Gold={reconciled_events:,} vs entrada Silver={total_input_events:,}"
)
assert gold_event_dist.filter(
    pl.col("event_code").is_null() | pl.col("turn").is_null() | pl.col("uf").is_null()
).height == 0, "Encontrados nulos em chaves de gold_event_distribution"
assert gold_event_dist.filter(
    (pl.col("percentage") < 0.0) | (pl.col("percentage") > 100.0)
).height == 0, "Percentuais fora do intervalo [0, 100] em gold_event_distribution"
print(f"✓ gold_event_distribution: {reconciled_events:,} eventos reconciliados 1:1.")

reconciled_sessions = int(gold_session_comp["session_count"].cast(pl.Int64).sum())
assert reconciled_sessions == df_sessions.height, (
    f"Divergência de contagem de sessões: soma na Gold={reconciled_sessions:,} vs entrada Silver={df_sessions.height:,}"
)
reconciled_session_events = int(gold_session_comp["total_events"].cast(pl.Int64).sum())
assert reconciled_session_events == total_input_events, (
    f"Divergência de total_events em gold_session_completeness: {reconciled_session_events:,} vs {total_input_events:,}"
)
assert gold_session_comp.filter(
    (pl.col("completion_rate") < 0.0) | (pl.col("completion_rate") > 100.0)
).height == 0, "Taxas de completude fora de [0, 100] em gold_session_completeness"
print(f"✓ gold_session_completeness: {reconciled_sessions:,} sessões e {reconciled_session_events:,} eventos reconciliados.")

reconciled_bio = int(gold_bio_attempts["attempt_count"].cast(pl.Int64).sum()) if gold_bio_attempts.height > 0 else 0
assert reconciled_bio == total_bio_rows, (
    f"Divergência de tentativas biométricas: soma na Gold={reconciled_bio:,} vs entrada Silver={total_bio_rows:,}"
)
if gold_bio_attempts.height > 0:
    assert gold_bio_attempts.filter(
        (pl.col("cumulative_share") < 0.0) | (pl.col("cumulative_share") > 100.001)
    ).height == 0, "Curva cumulativa fora de [0, 100] em gold_biometric_attempts"
print(f"✓ gold_biometric_attempts: {reconciled_bio:,} tentativas reconciliadas.")

dup_sec = (
    gold_section_summary.group_by(SESSION_KEY_COLS).len().filter(pl.col("len") > 1).height
)
assert dup_sec == 0, f"Chaves duplicadas em gold_section_turn_summary: {dup_sec}"
sec_total_events = int(gold_section_summary["total_events"].cast(pl.Int64).sum())
assert sec_total_events == total_input_events, (
    f"Divergência de total_events em gold_section_turn_summary: {sec_total_events:,} vs {total_input_events:,}"
)
drift_with_duration = gold_section_summary.filter(
    pl.col("has_clock_drift") & pl.col("observed_duration_sec").is_not_null()
).height
assert drift_with_duration == 0, (
    f"Guarda de clock drift violada: {drift_with_duration} seções com clock drift possuem observed_duration_sec não nulo"
)
for rate_col in ("biometric_recognition_rate_pct", "manual_enable_rate_pct", "turnout_rate_pct", "biometria_coverage_pct"):
    bad_rates = gold_section_summary.filter(
        pl.col(rate_col).is_not_null() & ((pl.col(rate_col) < 0.0) | (pl.col(rate_col) > 100.0))
    ).height
    assert bad_rates == 0, f"Taxa {rate_col} fora de [0, 100] em gold_section_turn_summary: {bad_rates}"
print(f"✓ gold_section_turn_summary: {gold_section_summary.height:,} seções × turno, 0 duplicatas, {sec_total_events:,} eventos.")

dup_mun = (
    gold_mun_summary.group_by(["election_year", "turn", "uf", "municipio_code"]).len().filter(pl.col("len") > 1).height
)
assert dup_mun == 0, f"Chaves duplicadas em gold_municipality_turn_summary: {dup_mun}"
assert int(gold_mun_summary["section_count"].cast(pl.Int64).sum()) == gold_section_summary.height, (
    "Soma de section_count em gold_municipality_turn_summary diverge de gold_section_turn_summary"
)
assert int(gold_mun_summary["total_events"].cast(pl.Int64).sum()) == total_input_events, (
    "Soma de total_events em gold_municipality_turn_summary diverge de fact_urna_event"
)
print(f"✓ gold_municipality_turn_summary: {gold_mun_summary.height:,} municípios × turno, 0 duplicatas.")

con = duckdb.connect(":memory:")
gold_tables = [
    ("gold_event_distribution", event_dist_path, gold_event_dist),
    ("gold_session_completeness", session_comp_path, gold_session_comp),
    ("gold_biometric_attempts", bio_attempts_path, gold_bio_attempts),
    ("gold_section_turn_summary", section_summary_path, gold_section_summary),
    ("gold_municipality_turn_summary", mun_summary_path, gold_mun_summary),
]
for tname, tpath, df_expected in gold_tables:
    fsize = tpath.stat().st_size
    assert fsize <= MAX_GOLD_FILE_BYTES, f"{tname} excedeu 128 MB: {fsize:,} bytes"
    duck_count = con.execute(f"SELECT count(*) FROM read_parquet('{tpath}')").fetchone()[0]
    assert duck_count == df_expected.height, f"DuckDB divergiu em {tname}: {duck_count} vs {df_expected.height}"
    print(f"✓ DuckDB verificou {tname:<32}: {duck_count:>10,} linhas ({fsize / (1024**2):>5.2f} MB)")
con.close()

print("\nTodas as asserções da camada Gold (5 tabelas) passaram com sucesso!")

Iniciando bateria de asserções e testes de consistência Gold (5 tabelas)...
✓ gold_event_distribution: 4,954,541,320 eventos reconciliados 1:1.
✓ gold_session_completeness: 998,355 sessões e 4,954,541,320 eventos reconciliados.
✓ gold_biometric_attempts: 887,549,299 tentativas reconciliadas.
✓ gold_section_turn_summary: 998,354 seções × turno, 0 duplicatas, 4,954,541,320 eventos.
✓ gold_municipality_turn_summary: 11,376 municípios × turno, 0 duplicatas.
✓ DuckDB verificou gold_event_distribution         :        832 linhas ( 0.01 MB)
✓ DuckDB verificou gold_session_completeness       :         57 linhas ( 0.01 MB)
✓ DuckDB verificou gold_biometric_attempts         :        216 linhas ( 0.01 MB)
✓ DuckDB verificou gold_section_turn_summary       :    998,354 linhas (46.87 MB)
✓ DuckDB verificou gold_municipality_turn_summary  :     11,376 linhas ( 0.61 MB)

Todas as asserções da camada Gold (5 tabelas) passaram com sucesso!


In [6]:
# ---------------------------------------------------------------------------
# Geração do Manifesto Gold e Publicação Opcional via KaggleHub SDK
# ---------------------------------------------------------------------------
manifest_data = {
    "generated_at": datetime.now(UTC).isoformat(),
    "pipeline_stage": "gold",
    "version": GOLD_VERSION,
    "tables": {
        "gold_event_distribution": {
            "path": str(event_dist_path.relative_to(GOLD_OUT.parent)),
            "rows": gold_event_dist.height,
            "bytes": event_dist_path.stat().st_size,
            "grain": "(event_code, turn, uf)",
            "columns": [{"name": c, "type": str(t)} for c, t in gold_event_dist.schema.items()],
        },
        "gold_session_completeness": {
            "path": str(session_comp_path.relative_to(GOLD_OUT.parent)),
            "rows": gold_session_comp.height,
            "bytes": session_comp_path.stat().st_size,
            "grain": "(turn, uf, final_state)",
            "columns": [{"name": c, "type": str(t)} for c, t in gold_session_comp.schema.items()],
        },
        "gold_biometric_attempts": {
            "path": str(bio_attempts_path.relative_to(GOLD_OUT.parent)),
            "rows": gold_bio_attempts.height,
            "bytes": bio_attempts_path.stat().st_size,
            "grain": "(turn, uf, biometric_attempt_num)",
            "columns": [{"name": c, "type": str(t)} for c, t in gold_bio_attempts.schema.items()],
        },
        "gold_section_turn_summary": {
            "path": str(section_summary_path.relative_to(GOLD_OUT.parent)),
            "rows": gold_section_summary.height,
            "bytes": section_summary_path.stat().st_size,
            "grain": "(election_year, turn, uf, municipio_code, zone_num, section_num)",
            "columns": [{"name": c, "type": str(t)} for c, t in gold_section_summary.schema.items()],
        },
        "gold_municipality_turn_summary": {
            "path": str(mun_summary_path.relative_to(GOLD_OUT.parent)),
            "rows": gold_mun_summary.height,
            "bytes": mun_summary_path.stat().st_size,
            "grain": "(election_year, turn, uf, municipio_code)",
            "columns": [{"name": c, "type": str(t)} for c, t in gold_mun_summary.schema.items()],
        },
    },
    "inputs_scanned": {
        "fact_urna_event": {"rows": total_input_events, "files_found": len(event_files)},
        "fact_voting_session": {"rows": df_sessions.height, "files_found": len(session_files)},
        "fact_boletim_urna": {"rows_joined": df_bu.height, "files_found": len(bu_files)},
        "dim_local_votacao_secao": {"rows_joined": df_loc.height, "files_found": len(loc_files)},
        "fact_eleitorado_perfil_secao": {"rows_joined": df_perfil.height, "files_found": len(perfil_files)},
        "bridge_correspondencia_efetivada": {"rows_joined": df_corresp.height, "files_found": len(corresp_files)},
        "fact_detalhe_votacao_secao": {"rows_joined": df_modelo.height, "files_found": len(detalhe_files)},
    },
    "status": "SUCCESS",
}

manifest_file = GOLD_OUT / "manifest.json"
_m_tmp = Path(STAGING_PATH) / "gold_manifest.json.tmp"
_m_tmp.parent.mkdir(parents=True, exist_ok=True)
_m_tmp.write_text(json.dumps(manifest_data, indent=2, ensure_ascii=False), encoding="utf-8")
if manifest_file.exists():
    manifest_file.unlink()
shutil.copyfile(_m_tmp, manifest_file)
_m_tmp.unlink(missing_ok=True)

print(f"Manifesto Gold gravado em: {manifest_file} ({manifest_file.stat().st_size:,} bytes)")

PHASE_ID = "08_gold_analytics"
PHASE_SUMMARY = (
    "Camada Gold v2.0.0: 5 produtos analíticos (eventos, completude, biometria, "
    "gold_section_turn_summary e gold_municipality_turn_summary)"
)
forbidden_extensions = {".zip", ".jez", ".tmp", ".part", ".log"}
offending_files = [
    str(p.relative_to(DATASET_PATH))
    for p in Path(DATASET_PATH).rglob("*")
    if p.is_file() and any(sfx.lower() in forbidden_extensions for sfx in p.suffixes)
]
if offending_files:
    raise ValueError(
        f"Upload abortado! Arquivos brutos/temporários detectados em DATASET_PATH: {offending_files[:10]}"
    )

if is_directory_empty(Path(DATASET_PATH)):
    raise RuntimeError(
        f"Upload abortado! DATASET_PATH ({DATASET_PATH}) está vazio após o processamento."
    )

if (
    globals().get("__name__") == "__main__"
    and not os.getenv("PYTEST_CURRENT_TEST")
    and not os.getenv("TSE_SKIP_KAGGLE_RELEASE")
):
    import kagglehub

    print(f"Publicando nova versão do dataset '{DATASET_HANDLE}' a partir de {DATASET_PATH}...")
    kagglehub.dataset_upload(
        handle=DATASET_HANDLE,
        local_dataset_dir=str(DATASET_PATH),
        version_notes=f"[{PHASE_ID}] {PHASE_SUMMARY}",
        ignore_patterns=[
            "*.tmp",
            "*.part",
            "*.zip",
            "*.jez",
            "*.log",
            "__pycache__/",
            ".ipynb_checkpoints/",
            ".DS_Store",
        ],
    )
    print(f"[SUCESSO] Nova versão de '{DATASET_HANDLE}' publicada no Kaggle!")

Manifesto Gold gravado em: /content/drive/MyDrive/tse-eleicoes-dados/data/gold/manifest.json (10,642 bytes)


## Definição das Métricas, Regras e Limitações

### 1. Catálogo de Produtos Gold (5 Tabelas)

| Tabela Gold | Grão | Principais Métricas | Propósito Analítico |
|---|---|---|---|
| `gold_event_distribution` | `(event_code, turn, uf)` | `event_count`, `total_uf_events`, `percentage`, `clock_drift_count` | Prevalência de eventos e incidência de *clock drift* por UF e turno. |
| `gold_session_completeness` | `(turn, uf, final_state)` | `session_count`, `completion_rate`, `contingency_session_count`, `total_voter_cycles`, `total_events` | Saúde operacional, contingências e encerramento regular de urnas. |
| `gold_biometric_attempts` | `(turn, uf, biometric_attempt_num)` | `attempt_count`, `percentage`, `cumulative_count`, `cumulative_share` | Curva de resolução biométrica por tentativa sucessiva (1ª, 2ª, 3ª, 4ª+). |
| `gold_section_turn_summary` | `(election_year, turn, uf, municipio_code, zone_num, section_num)` | 46 colunas combinando logs Silver + BU oficial (`CD_CARGO_PERGUNTA=1`) + eleitorado + local + modelo + correspondência | Fato analítico central sub-100ms para o **TS-Zé** e estudos eleitorais no grão seção × turno. |
| `gold_municipality_turn_summary` | `(election_year, turn, uf, municipio_code)` | 27 colunas agregadas a partir de `gold_section_turn_summary` | Rankings municipais, mapas e painéis regionais instantâneos. |

### 2. Regras e Ressalvas Metodológicas
- **Comparecimento oficial 100% nacional (1º e 2º turnos):** extraído de `fact_boletim_urna` filtrado em `CD_CARGO_PERGUNTA = 1` (Presidente), presente em todas as 28 UFs (incluindo `ZZ`) nos dois turnos. `fact_detalhe_votacao_secao` (que cobre apenas cargos estaduais) é usado exclusivamente para enriquecer o modelo de hardware (`urna_model_desc`).
- **Guarda contra *clock drift* de BIOS:** quando `has_clock_drift = True`, `observed_duration_sec` é mantido como `NULL` para evitar distorções de duração causadas por relógios reiniciados em `01/01/1970`.
- **Sem filtros excludentes de eventos:** todos os eventos classificados e `UNKNOWN_EVENT` são contabilizados para conservação exata 1:1 com `silver/fact_urna_event`.